# 06 — Convergence, ECDFs and Strategy Comparisons

Calculate exploratory adaptive-target profiles from benchmark traces. Match analysis settings to the benchmark protocol; saved outputs may be historical. Run only the sections needed. PNG exports are explicit and not cached.


In [1]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "shared").is_dir()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project directory or notebooks/analysis/.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))

# Notebook-owned presentation helpers are imported from the project root.
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import plotly.io as pio
pio.defaults.mathjax = None
from benchmark.infra.di import BenchmarkProvider
from benchmark.application.benchmark_config import BenchmarkConfig, ReliabilityConfig
from benchmark.infra.storage.analysis_results_store import AnalysisResultsStore
from shared.infra.lifecycle import WorkflowScope
from benchmark.application.analysis.analyze_ecdf_and_convergence import AnalyzeEcdfAndConvergence
from benchmark.application.analysis.analyze_performance import AnalyzePerformance
from shared.config import RESULTS_DIR
from notebooks.analysis.plotting.exporter import AnalysisFigureExporter


figure_exporter = AnalysisFigureExporter(RESULTS_DIR)

# Optional condition filters; None selects the full available matrix.
FILTER_DIMS = None
FILTER_PROBLEMS = None
FILTER_NOISE_STDS = None


analysis_config = BenchmarkConfig(
    budget=100_000, trials_per_condition=20, random_seed=42, first_instance_id=2,
    trial_timeout_seconds=30.0, observation_trace_points=64,
    classical_baselines=["cmaes", "de", "pso"],
    cross_function_enabled=False,
    cross_function_problem_ids=[1, 8, 11, 15, 21],
    reliability=ReliabilityConfig(
        primary_prompt_strategy="baseline", primary_target=1e-8, secondary_target=1e-2,
        bootstrap_samples=1000, bootstrap_seed=20260923,
        checkpoint_fractions=[0.01, 0.1, 0.5, 1.0],
    ),
)

# Prepare services once; infrastructure owns shutdown cleanup.
if "services" in globals():
    services.close()
services = WorkflowScope(BenchmarkProvider(), context={}, request_context={BenchmarkConfig: analysis_config})
profile_analysis = services.get(AnalyzeEcdfAndConvergence)
performance_analysis = services.get(AnalyzePerformance)
results_store = services.get(AnalysisResultsStore)


Diagnostic log: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/logs/benchmark/2026-11-23_18-45-29_UTC-17975afb/diagnostics.log


In [2]:
performance_result = performance_analysis.execute(dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
performance = performance_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if performance is not None:
    performance_snapshot = results_store.save(performance)
    print(f"Calculated exploratory adaptive-target metrics for {len(performance.table)} entries.")
    
    print(f"Saved numerical results: {performance_snapshot}")


Calculated exploratory adaptive-target metrics for 1776 entries.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/performance/598e0f3944aae3f05b7f519546a6a2a8943425cc64f5a0ef6e490233cc51a3fe


## 1. Exploratory model comparison


In [3]:
if performance is not None:
    figure_exporter.export_model_scale(performance)
else:
    print("Complete the preceding workflow successfully before using its data.")


## 2. Strategy comparison by landscape hardness


In [4]:
if performance is not None:
    figure_exporter.export_hardness_ablation(performance)
else:
    print("Complete the preceding workflow successfully before using its data.")


## 3. Explicit convergence and ECDFs


In [5]:
explicit_profiles_result = profile_analysis.execute(mode="explicit", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
explicit_profiles = explicit_profiles_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if explicit_profiles is not None:
    explicit_profiles_snapshot = results_store.save(explicit_profiles)
    print(f"Calculated {len(explicit_profiles.conditions)} explicit condition profiles; no PNGs exported yet.")
    
    print(f"Saved numerical results: {explicit_profiles_snapshot}")


Calculated 40 explicit condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/explicit/2df8312f2d95790e7c761b3c8a5f525af0ebcc9c35471e6ceff820b6c85c9d2e


In [6]:
if explicit_profiles is not None:
    figure_exporter.export_profiles(explicit_profiles)
else:
    print("Complete the preceding workflow successfully before using its data.")


## 4. Implicit convergence and ECDFs


In [7]:
implicit_profiles_result = profile_analysis.execute(mode="implicit", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
implicit_profiles = implicit_profiles_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if implicit_profiles is not None:
    implicit_profiles_snapshot = results_store.save(implicit_profiles)
    print(f"Calculated {len(implicit_profiles.conditions)} implicit condition profiles; no PNGs exported yet.")
    
    print(f"Saved numerical results: {implicit_profiles_snapshot}")


Calculated 29 implicit condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/implicit/795dcfaf3e3c41af6f634a12fde482edb57bf14d4355b9cc214586007df8842d


In [8]:
if implicit_profiles is not None:
    figure_exporter.export_profiles(implicit_profiles)
else:
    print("Complete the preceding workflow successfully before using its data.")


## 5. Explicit versus implicit comparison


In [9]:
comparison_profiles_result = profile_analysis.execute(mode="comparison", dims=FILTER_DIMS, problems=FILTER_PROBLEMS, noise_stds=FILTER_NOISE_STDS)
comparison_profiles = comparison_profiles_result.on(
    on_success=lambda value: value,
    on_failure=lambda error: print(f"Error: {error.message}"),
)
if comparison_profiles is not None:
    comparison_profiles_snapshot = results_store.save(comparison_profiles)
    print(f"Calculated {len(comparison_profiles.conditions)} comparison condition profiles; no PNGs exported yet.")
    
    print(f"Saved numerical results: {comparison_profiles_snapshot}")


Calculated 29 comparison condition profiles; no PNGs exported yet.
Saved numerical results: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/results/analysis/profiles/comparison/6f71f707afbb51985c51d5e4cbc3b129c4fd0434970a6b66d624e6c90ec576a6


In [10]:
if comparison_profiles is not None:
    figure_exporter.export_profiles(comparison_profiles)
else:
    print("Complete the preceding workflow successfully before using its data.")
